# Bangla License Plate OCR Workflow

This notebook uses DeblurGAN-restored license plate crops as the input dataset for Bengali OCR. The direct YOLOv8-crop OCR experiment is saved separately under `runs/ocr_yolov8_input_token_ocr/`; this notebook is switched back to the DeblurGAN-output branch so the OCR model can be trained and evaluated again on restored crops.


In [ ]:
# Optional one-time installs if your notebook environment is missing OCR packages.
# %pip install easyocr pytesseract tqdm pandas matplotlib opencv-python pillow

# Tesseract also needs the system application plus Bengali traineddata.
# Windows example after installing Tesseract OCR:
# import pytesseract
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"



In [ ]:
from pathlib import Path
import os
import shutil
from collections import Counter
import ast
import json
import math
import random
import re
import unicodedata
import warnings

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else []

try:
    from PIL import Image, ImageDraw, ImageFont
except Exception:
    Image = ImageDraw = ImageFont = None

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
PROJECT_ROOT = Path.cwd()
OCR_LABEL_DATASET = PROJECT_ROOT / "dataset"
DEBLURGAN_PAIR_ROOT = PROJECT_ROOT / "deblurgan_data" / "ocr_labeled_paired_plate_crops_psnr_v2"
DEBLURGAN_CHECKPOINT = PROJECT_ROOT / "runs" / "deblurgan_clean_dataset_irv2_fpn_psnr_v2" / "checkpoints" / "best_generator.pt"

OCR_RUN_ROOT = PROJECT_ROOT / "runs" / "ocr_text_recognition_psnr_v2"
MANIFEST_ROOT = OCR_RUN_ROOT / "manifests"
DEBLURRED_OCR_INPUT_ROOT = OCR_RUN_ROOT / "deblurred_ocr_inputs"
PREDICTION_ROOT = OCR_RUN_ROOT / "predictions"
FIGURE_ROOT = OCR_RUN_ROOT / "figures"
YOLO_TOKEN_DATASET_ROOT = OCR_RUN_ROOT / "yolo_token_dataset"
YOLO_TOKEN_RUN_ROOT = OCR_RUN_ROOT / "yolo_token_ocr"
YOLO_TOKEN_RUN_NAME = "yolov8_token_detector"
YOLO_TOKEN_WEIGHTS = YOLO_TOKEN_RUN_ROOT / YOLO_TOKEN_RUN_NAME / "weights" / "best.pt"
YOLO_TOKEN_BASE_MODEL = PROJECT_ROOT / "yolov8s.pt"
ULTRALYTICS_CONFIG_DIR = OCR_RUN_ROOT / "ultralytics_config"

for folder in [MANIFEST_ROOT, DEBLURRED_OCR_INPUT_ROOT, PREDICTION_ROOT, FIGURE_ROOT, YOLO_TOKEN_DATASET_ROOT, YOLO_TOKEN_RUN_ROOT, ULTRALYTICS_CONFIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "valid", "test"]
PLATE_HEIGHT = 96
PLATE_WIDTH = 256
HAS_OCR_GROUND_TRUTH = True
REQUIRE_VALID_OCR_LABELS = True

# OCR consumes DeblurGAN-restored plate crops.
OCR_IMAGE_VARIANTS = {
    "deblurred": "ocr_image",
}
OCR_EVALUATION_VARIANTS = ["deblurred"]
DEPLOYABLE_OCR_VARIANTS = ["deblurred"]
DEFAULT_DEPLOYABLE_OCR_VARIANT = "deblurred"

BOTTOM_DIGIT_COUNT = 6
OVERWRITE_DEBLURRED_OCR_INPUTS = False
OVERWRITE_YOLO_TOKEN_DATASET_IMAGES = True

YOLO_TOKEN_IMAGE_SIZE = 320
YOLO_TOKEN_EPOCHS = 100
YOLO_TOKEN_BATCH_SIZE = 16
YOLO_TOKEN_CONFIDENCE = 0.20
YOLO_TOKEN_IOU = 0.45
RUN_YOLO_TOKEN_OCR_TRAINING = True
RETRAIN_YOLO_TOKEN_OCR = True

EASYOCR_LANGS = ["bn", "en"]
TESSERACT_LANG = "ben"
TESSERACT_CONFIG = "--oem 1 --psm 7"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project root:", PROJECT_ROOT)
print("OCR label dataset:", OCR_LABEL_DATASET)
print("DeblurGAN paired crop dataset:", DEBLURGAN_PAIR_ROOT)
print("DeblurGAN generator checkpoint:", DEBLURGAN_CHECKPOINT)
print("DeblurGAN OCR input root:", DEBLURRED_OCR_INPUT_ROOT)
print("OCR run root:", OCR_RUN_ROOT)
print("YOLO token base model:", YOLO_TOKEN_BASE_MODEL)
print("YOLO token weights:", YOLO_TOKEN_WEIGHTS)
print("OCR ground truth available:", HAS_OCR_GROUND_TRUTH)
print("Torch device:", DEVICE)


In [ ]:
def project_path(value):
    text = str(value).replace("\\", "/")
    path = Path(text)
    return path if path.is_absolute() else PROJECT_ROOT / path


def relative_text(path):
    path = Path(path)
    try:
        return path.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return path.as_posix()


def require_path(path, label):
    if not Path(path).exists():
        raise FileNotFoundError(f"Missing {label}: {path}")


require_path(OCR_LABEL_DATASET, "character-labeled OCR dataset")
require_path(OCR_LABEL_DATASET / "data.yaml", "OCR dataset data.yaml")
require_path(DEBLURGAN_PAIR_ROOT, "DeblurGAN OCR paired crop dataset")
require_path(DEBLURGAN_CHECKPOINT, "trained DeblurGAN generator checkpoint")
print("Using DeblurGAN output as OCR input:", DEBLURRED_OCR_INPUT_ROOT)


## 1. Build OCR Label Manifest

This section reads the character-level labels in `dataset/`, maps the romanized YOLO classes to Bengali plate text, and prepares the target text. The actual OCR input crop is created later by running the trained YOLOv8 plate detector.


In [ ]:
BANGLA_DIGITS = {
    "0": "\u09e6", "1": "\u09e7", "2": "\u09e8", "3": "\u09e9", "4": "\u09ea",
    "5": "\u09eb", "6": "\u09ec", "7": "\u09ed", "8": "\u09ee", "9": "\u09ef",
}

TARGET_TOKEN_MAP = {
    **BANGLA_DIGITS,
    "dhaka": "\u09a2\u09be\u0995\u09be",
    "daka": "\u09a2\u09be\u0995\u09be",
    "sylhet": "\u09b8\u09bf\u09b2\u09c7\u099f",
    "silhet": "\u09b8\u09bf\u09b2\u09c7\u099f",
    "chattro": "\u099a\u099f\u09cd\u099f",
    "chattogram": "\u099a\u099f\u09cd\u099f",
    "chittagong": "\u099a\u099f\u09cd\u099f",
    "metro": "\u09ae\u09c7\u099f\u09cd\u09b0\u09cb",
    "ka": "\u0995",
    "kha": "\u0996",
    "ga": "\u0997",
    "gha": "\u0998",
    "cha": "\u099a",
    "chha": "\u099b",
    "ja": "\u099c",
    "jha": "\u099d",
    "Ta": "\u099f",
    "Tha": "\u09a0",
    "ta": "\u099f",
    "tha": "\u09a0",
    "ba": "\u09ac",
    "bha": "\u09ad",
    "m": "\u09ae",
    "ma": "\u09ae",
    "na": "\u09a8",
    "sa": "\u09b8",
}

TARGET_TOKEN_ALIASES = {str(key).strip().lower(): value for key, value in TARGET_TOKEN_MAP.items()}
BANGLA_DIGIT_VALUES = set(BANGLA_DIGITS.values())
FULL_PLATE_CLASSES = {"license-plate-number", "license_plate", "license-plate", "license plate", "license_plate_number"}


def bangla_token_for_class(class_name):
    key = str(class_name).strip().lower()
    if key in FULL_PLATE_CLASSES:
        return None
    return TARGET_TOKEN_ALIASES.get(key)


def load_yolo_class_names(data_yaml_path):
    text = Path(data_yaml_path).read_text(encoding="utf-8")
    inline_match = re.search(r"names:\s*(\[[\s\S]*?\])", text)
    if inline_match:
        return list(ast.literal_eval(inline_match.group(1)))

    names = {}
    in_names = False
    for line in text.splitlines():
        if line.strip().startswith("names:"):
            in_names = True
            continue
        if in_names:
            if not line.startswith(" ") and not line.startswith("\t"):
                break
            match = re.match(r"\s*(\d+)\s*:\s*(.+?)\s*$", line)
            if match:
                names[int(match.group(1))] = match.group(2).strip().strip("'\"")
    if names:
        return [names[index] for index in sorted(names)]
    raise ValueError(f"Could not parse class names from {data_yaml_path}")


CLASS_NAMES = load_yolo_class_names(OCR_LABEL_DATASET / "data.yaml")
print("OCR dataset class count:", len(CLASS_NAMES))
print("Bengali OCR token classes kept:", sorted(TARGET_TOKEN_ALIASES))


def read_character_tokens(label_path, class_names=CLASS_NAMES):
    tokens = []
    label_path = Path(label_path)
    if not label_path.exists():
        return tokens
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        try:
            class_id = int(float(parts[0]))
            x_center, y_center, width, height = map(float, parts[1:5])
        except ValueError:
            continue
        if class_id < 0 or class_id >= len(class_names):
            continue
        class_name = str(class_names[class_id]).strip()
        token = bangla_token_for_class(class_name)
        if token is None:
            continue
        tokens.append({
            "token": token,
            "class_name": class_name,
            "x_center": x_center,
            "y_center": y_center,
            "width": width,
            "height": height,
        })
    return tokens


def split_token_rows(tokens):
    if not tokens:
        return []
    median_height = float(np.median([item["height"] for item in tokens]))
    row_threshold = max(0.10, median_height * 0.75)
    rows = []
    for item in sorted(tokens, key=lambda value: value["y_center"]):
        best_index = None
        best_distance = None
        for index, row in enumerate(rows):
            row_y = float(np.mean([entry["y_center"] for entry in row]))
            distance = abs(item["y_center"] - row_y)
            if distance <= row_threshold and (best_distance is None or distance < best_distance):
                best_index = index
                best_distance = distance
        if best_index is None:
            rows.append([item])
        else:
            rows[best_index].append(item)
    rows.sort(key=lambda row: float(np.mean([entry["y_center"] for entry in row])))
    for row in rows:
        row.sort(key=lambda value: value["x_center"])
    return rows


def join_row_tokens(tokens):
    pieces = []
    digit_buffer = ""
    for token in tokens:
        if all(char in BANGLA_DIGIT_VALUES for char in token):
            digit_buffer += token
        else:
            if digit_buffer:
                pieces.append(digit_buffer)
                digit_buffer = ""
            pieces.append(token)
    if digit_buffer:
        pieces.append(digit_buffer)
    return " ".join(pieces)


def split_single_row_plate_text(text):
    tokens = str(text).split()
    digit_start = len(tokens)
    for index, token in enumerate(tokens):
        if all(char in BANGLA_DIGIT_VALUES for char in token):
            digit_start = index
            break
    top_text = " ".join(tokens[:digit_start]).strip()
    bottom_text = " ".join(tokens[digit_start:]).strip()
    return top_text, bottom_text


def plate_rows_from_label(label_path):
    rows = split_token_rows(read_character_tokens(label_path))
    return [join_row_tokens([item["token"] for item in row]) for row in rows]


def plate_text_from_label(label_path):
    return " ".join(plate_rows_from_label(label_path)).strip()


CITY_TOKENS = [TARGET_TOKEN_ALIASES["dhaka"], TARGET_TOKEN_ALIASES["chattro"], TARGET_TOKEN_ALIASES["sylhet"]]
METRO_TOKEN = TARGET_TOKEN_ALIASES["metro"]
SERIES_TOKEN_KEYS = ["ka", "kha", "ga", "gha", "cha", "chha", "ja", "jha", "ta", "tha", "ba", "bha", "m", "na", "sa"]
SERIES_TOKENS = {TARGET_TOKEN_ALIASES[key] for key in SERIES_TOKEN_KEYS}


def token_is_digit_group(token):
    return bool(token) and all(char in BANGLA_DIGIT_VALUES for char in token)


def clean_plate_fields_from_text(text):
    tokens = [token for token in str(text).split() if token]
    city = next((token for token in tokens if token in CITY_TOKENS), "")
    metro = METRO_TOKEN if METRO_TOKEN in tokens else ""
    series_candidates = [token for token in tokens if token in SERIES_TOKENS]
    series = series_candidates[-1] if series_candidates else ""
    digits = "".join(char for token in tokens for char in token if char in BANGLA_DIGIT_VALUES)

    top_parts = [part for part in [city, metro, series] if part]
    top_text = " ".join(top_parts).strip()
    bottom_text = digits.strip()
    return top_text, bottom_text


def split_plate_rows_for_ocr(row_texts):
    row_texts = [text.strip() for text in row_texts if str(text).strip()]
    if not row_texts:
        return "", "", 0

    raw_text = " ".join(row_texts).strip()
    clean_top, clean_bottom = clean_plate_fields_from_text(raw_text)
    if clean_top and clean_bottom:
        return clean_top, clean_bottom, len(row_texts)

    if len(row_texts) == 1:
        top_text, bottom_text = split_single_row_plate_text(row_texts[0])
        return top_text, bottom_text, 1
    top_text = " ".join(row_texts[:-1]).strip()
    bottom_text = row_texts[-1].strip()
    return top_text, bottom_text, len(row_texts)


def has_valid_top_components(top_text):
    tokens = [token for token in str(top_text).split() if token]
    return any(token in CITY_TOKENS for token in tokens) and any(token in SERIES_TOKENS for token in tokens)


def is_valid_plate_label(top_text, bottom_text):
    digits = "".join(char for char in str(bottom_text) if char in BANGLA_DIGIT_VALUES)
    return has_valid_top_components(top_text) and len(digits) == BOTTOM_DIGIT_COUNT


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def image_path_for_label(split, label_path):
    image_dir = OCR_LABEL_DATASET / split / "images"
    for extension in IMAGE_EXTENSIONS:
        candidate = image_dir / f"{label_path.stem}{extension}"
        if candidate.exists():
            return candidate
    matches = list(image_dir.glob(f"{label_path.stem}.*"))
    matches = [path for path in matches if path.suffix.lower() in IMAGE_EXTENSIONS]
    return matches[0] if matches else None


def source_image_size(image_path):
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")
    height, width = image.shape[:2]
    return width, height


def build_ocr_manifest(split):
    pair_manifest_path = DEBLURGAN_PAIR_ROOT / split / "manifest.csv"
    require_path(pair_manifest_path, f"DeblurGAN {split} paired crop manifest")
    pair_frame = pd.read_csv(pair_manifest_path)

    records = []
    skipped = 0
    missing_deblurred = 0

    for pair in pair_frame.itertuples(index=False):
        source_image = project_path(pair.source_image)
        source_label = project_path(pair.source_label)
        blurry_crop = project_path(pair.blurry)
        sharp_crop = project_path(pair.sharp)

        if not source_image.exists() or not source_label.exists() or not blurry_crop.exists():
            skipped += 1
            continue

        row_texts = plate_rows_from_label(source_label)
        raw_target_text = " ".join(row_texts).strip()
        top_text, bottom_text, row_count = split_plate_rows_for_ocr(row_texts)
        bottom_text = "".join(char for char in bottom_text if char in BANGLA_DIGIT_VALUES)
        target_text = " ".join(part for part in [top_text, bottom_text] if part).strip()

        if REQUIRE_VALID_OCR_LABELS and not is_valid_plate_label(top_text, bottom_text):
            skipped += 1
            continue

        deblurred_path = DEBLURRED_OCR_INPUT_ROOT / split / f"{blurry_crop.stem}_deblurred.png"
        if not deblurred_path.exists():
            missing_deblurred += 1

        records.append({
            "split": split,
            "image_id": source_image.stem,
            "variant": int(getattr(pair, "variant", 0)),
            "target_text": target_text,
            "raw_target_text": raw_target_text,
            "top_text": top_text,
            "bottom_text": bottom_text,
            "row_count": row_count if row_count > 0 else 2,
            "source_image": relative_text(source_image),
            "source_label": relative_text(source_label),
            "blurry_crop": relative_text(blurry_crop),
            "sharp_crop": relative_text(sharp_crop),
            "ocr_image": relative_text(deblurred_path),
            "x1": float(pair.x1),
            "y1": float(pair.y1),
            "x2": float(pair.x2),
            "y2": float(pair.y2),
            "input_source": "deblurgan_restored_crop",
        })

    frame = pd.DataFrame(records)
    frame.to_csv(MANIFEST_ROOT / f"{split}_ocr_manifest.csv", index=False, encoding="utf-8-sig")
    print(
        f"{split}: prepared {len(frame)} DeblurGAN OCR rows, "
        f"skipped {skipped} invalid/missing rows, "
        f"missing deblurred crops {missing_deblurred}"
    )
    return frame


ocr_manifests = {split: build_ocr_manifest(split) for split in SPLITS}

all_manifest_rows = pd.concat(ocr_manifests.values(), ignore_index=True)
print("OCR rows with Bengali text labels:", len(all_manifest_rows))
display(all_manifest_rows.head(10)[["split", "image_id", "target_text", "top_text", "bottom_text", "ocr_image"]])


In [ ]:
def show_target_examples(frame, count=6):
    sample = frame.sample(min(count, len(frame)), random_state=7) if len(frame) else frame
    if sample.empty:
        print("No samples to show.")
        return
    figure, axes = plt.subplots(len(sample), 2, figsize=(10, 2.8 * len(sample)))
    if len(sample) == 1:
        axes = np.array([axes])
    for axis_row, row in zip(axes, sample.itertuples(index=False)):
        image = cv2.imread(str(project_path(row.source_image)))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        axis_row[0].imshow(image)
        axis_row[0].set_title("Source image before YOLOv8 crop")
        axis_row[0].axis("off")
        axis_row[1].text(0.02, 0.6, row.target_text, fontsize=18)
        axis_row[1].set_title("Reconstructed OCR target")
        axis_row[1].axis("off")
    plt.tight_layout()
    plt.show()


show_target_examples(ocr_manifests["test"])


## 2. Use DeblurGAN OCR Inputs

This section verifies that the OCR input images are DeblurGAN-restored plate crops. These images are saved under `runs/ocr_text_recognition_psnr_v2/deblurred_ocr_inputs/` and are used as the input dataset for EasyOCR, Tesseract, and the YOLOv8 token OCR model.


In [ ]:
def load_ultralytics_yolo():
    os.environ.setdefault("YOLO_CONFIG_DIR", str(ULTRALYTICS_CONFIG_DIR))
    from ultralytics import YOLO
    return YOLO


def load_yolov8_plate_detector():
    YOLO = load_ultralytics_yolo()
    return YOLO(str(YOLO_PLATE_WEIGHTS))


def box_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = map(float, box_a)
    bx1, by1, bx2, by2 = map(float, box_b)
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    intersection = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0


def clip_box_to_image(box, image_shape):
    height, width = image_shape[:2]
    x1, y1, x2, y2 = map(float, box)
    return [
        max(0.0, min(float(width - 1), x1)),
        max(0.0, min(float(height - 1), y1)),
        max(0.0, min(float(width - 1), x2)),
        max(0.0, min(float(height - 1), y2)),
    ]


def crop_plate_from_box(image, box):
    x1, y1, x2, y2 = clip_box_to_image(box, image.shape)
    x1, y1, x2, y2 = map(int, [round(x1), round(y1), round(x2), round(y2)])
    if x2 <= x1 or y2 <= y1:
        return None
    crop = image[y1:y2, x1:x2]
    if crop.size == 0:
        return None
    return cv2.resize(crop, (PLATE_WIDTH, PLATE_HEIGHT), interpolation=cv2.INTER_AREA)


def detections_from_plate_result(result):
    detections = []
    if result is None or result.boxes is None:
        return detections
    xyxy = result.boxes.xyxy.detach().cpu().numpy()
    confidences = result.boxes.conf.detach().cpu().numpy()
    for box, confidence in zip(xyxy, confidences):
        detections.append({"box": [float(value) for value in box], "confidence": float(confidence)})
    return detections


def choose_plate_detection(detections, label_box):
    if not detections:
        return None
    return max(
        detections,
        key=lambda item: (box_iou(item["box"], label_box), item["confidence"]),
    )


In [ ]:
def verify_deblurgan_ocr_inputs(manifests):
    verified_manifests = {}
    missing = []

    for split, frame in manifests.items():
        records = []
        for row in frame.itertuples(index=False):
            row_dict = row._asdict()
            ocr_image = project_path(row.ocr_image)
            if not ocr_image.exists():
                missing.append(relative_text(ocr_image))
                continue
            row_dict["input_source"] = "deblurgan_restored_crop"
            records.append(row_dict)

        verified = pd.DataFrame(records)
        verified.to_csv(MANIFEST_ROOT / f"{split}_ocr_manifest.csv", index=False, encoding="utf-8-sig")
        verified_manifests[split] = verified
        print(f"{split}: {len(verified)} DeblurGAN-restored OCR inputs verified")

    if missing:
        preview = "\n".join(missing[:10])
        raise FileNotFoundError(
            f"Missing {len(missing)} DeblurGAN-restored OCR input crops. "
            "Run deblurgan_workflow.ipynb first, then rerun this OCR notebook. "
            f"First missing files:\n{preview}"
        )

    return verified_manifests


ocr_manifests = verify_deblurgan_ocr_inputs(ocr_manifests)

missing = []
for split, frame in ocr_manifests.items():
    missing.extend([path for path in frame["ocr_image"] if not project_path(path).exists()])
print("Missing DeblurGAN OCR input crops:", len(missing))
assert not missing, "All OCR inputs must be DeblurGAN-restored crops."


In [ ]:
def show_deblurgan_ocr_inputs(frame, count=5):
    sample = frame.sample(min(count, len(frame)), random_state=11) if len(frame) else frame
    if sample.empty:
        print("No OCR inputs to show.")
        return
    figure, axes = plt.subplots(len(sample), 2, figsize=(10, 2.8 * len(sample)))
    if len(sample) == 1:
        axes = np.array([axes])
    for axis_row, row in zip(axes, sample.itertuples(index=False)):
        for axis, path, title in zip(
            axis_row,
            [row.blurry_crop, row.ocr_image],
            ["Blurry crop", "DeblurGAN OCR input crop"],
        ):
            image = cv2.imread(str(project_path(path)))
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            axis.imshow(image)
            axis.set_title(title)
            axis.axis("off")
    plt.tight_layout()
    plt.show()


show_deblurgan_ocr_inputs(ocr_manifests["test"])


## 3. OCR Evaluation Helpers

CER is normalized character edit distance. WER is normalized token edit distance after whitespace normalization. Exact plate match accuracy is the fraction of samples where the normalized prediction exactly equals the normalized target.



In [ ]:
ASCII_TO_BANGLA_DIGIT = str.maketrans(BANGLA_DIGITS)
PUNCTUATION_PATTERN = re.compile(r"[-_:,.;/\|()\[\]{}]+")

ROMANIZED_TOKEN_TO_BANGLA = {
    key: value
    for key, value in TARGET_TOKEN_ALIASES.items()
    if key not in BANGLA_DIGITS
}
ROMANIZED_TOKEN_TO_BANGLA.update({
    "ctg": TARGET_TOKEN_ALIASES["chattro"],
    "choto": TARGET_TOKEN_ALIASES["chattro"],
    "chottro": TARGET_TOKEN_ALIASES["chattro"],
    "mtr": METRO_TOKEN,
    "met": METRO_TOKEN,
    "g": TARGET_TOKEN_ALIASES["ga"],
    "k": TARGET_TOKEN_ALIASES["ka"],
    "c": TARGET_TOKEN_ALIASES["cha"],
    "j": TARGET_TOKEN_ALIASES["ja"],
    "b": TARGET_TOKEN_ALIASES["ba"],
    "n": TARGET_TOKEN_ALIASES["na"],
    "s": TARGET_TOKEN_ALIASES["sa"],
})
OCR_ALLOWLIST = "".join(sorted(set("".join(ROMANIZED_TOKEN_TO_BANGLA.values()) + "".join(BANGLA_DIGITS.values()))))


def normalize_text(text):
    if text is None or (isinstance(text, float) and math.isnan(text)):
        text = ""
    text = unicodedata.normalize("NFC", str(text))
    text = text.translate(ASCII_TO_BANGLA_DIGIT)
    text = text.replace("\n", " ").replace("\r", " ")
    text = PUNCTUATION_PATTERN.sub(" ", text)
    kept = []
    for char in text:
        if char.isspace():
            kept.append(" ")
        elif "\u0980" <= char <= "\u09ff":
            kept.append(char)
        elif char.isalnum():
            kept.append(char.lower())
    tokens = "".join(kept).split()
    converted_tokens = [ROMANIZED_TOKEN_TO_BANGLA.get(token.lower(), token) for token in tokens]
    return " ".join(converted_tokens)


def edit_distance(reference, hypothesis):
    reference = list(reference)
    hypothesis = list(hypothesis)
    previous = list(range(len(hypothesis) + 1))
    for i, ref_item in enumerate(reference, start=1):
        current = [i]
        for j, hyp_item in enumerate(hypothesis, start=1):
            cost = 0 if ref_item == hyp_item else 1
            current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + cost))
        previous = current
    return previous[-1]


def character_error_rate(reference, hypothesis):
    reference = normalize_text(reference)
    hypothesis = normalize_text(hypothesis)
    return edit_distance(reference, hypothesis) / max(1, len(reference))


def word_error_rate(reference, hypothesis):
    reference_words = normalize_text(reference).split()
    hypothesis_words = normalize_text(hypothesis).split()
    return edit_distance(reference_words, hypothesis_words) / max(1, len(reference_words))


def evaluate_predictions(predictions, method_name):
    records = []
    for row in predictions.itertuples(index=False):
        reference = normalize_text(getattr(row, "target_text", ""))
        prediction = normalize_text(getattr(row, "prediction", ""))
        has_reference = bool(reference) and HAS_OCR_GROUND_TRUTH
        records.append({
            "method": method_name,
            "split": row.split,
            "image_id": row.image_id,
            "target_text": getattr(row, "target_text", ""),
            "prediction": prediction,
            "normalized_target": reference,
            "normalized_prediction": prediction,
            "cer": character_error_rate(reference, prediction) if has_reference else np.nan,
            "wer": word_error_rate(reference, prediction) if has_reference else np.nan,
            "exact_match": int(reference == prediction) if has_reference else np.nan,
            "ocr_image": row.ocr_image,
        })
    return pd.DataFrame(records)


def empty_predictions(frame, message):
    result = frame[["split", "image_id", "target_text", "ocr_image"]].copy()
    result["prediction"] = ""
    result["note"] = message
    return result


## 4. Main OCR: EasyOCR

EasyOCR runs on the DeblurGAN-restored OCR crop. The first run may download EasyOCR model weights in your notebook environment.


In [ ]:
def crop_ocr_row_image(image, field, row_count=2):
    height, width = image.shape[:2]
    if row_count <= 1:
        return image
    if field == "top":
        y1, y2 = 0, int(height * 0.66)
    elif field == "bottom":
        y1, y2 = int(height * 0.40), height
    else:
        y1, y2 = 0, height
    return image[max(0, y1):min(height, y2), :]


def run_easyocr(frame):
    try:
        import easyocr
    except Exception as exc:
        print("EasyOCR is not available:", exc)
        return empty_predictions(frame, "easyocr import failed")

    reader = easyocr.Reader(EASYOCR_LANGS, gpu=torch.cuda.is_available())
    records = []
    for row in tqdm(list(frame.itertuples(index=False)), desc="EasyOCR row-wise test"):
        image_path = project_path(row.ocr_image)
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            prediction = ""
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            parts = []
            for field in ["top", "bottom"]:
                row_image = crop_ocr_row_image(image, field, getattr(row, "row_count", 2))
                try:
                    result = reader.readtext(
                        row_image,
                        detail=0,
                        paragraph=False,
                        decoder="beamsearch",
                        allowlist=OCR_ALLOWLIST,
                    )
                    parts.append(" ".join(str(part) for part in result))
                except Exception as exc:
                    parts.append("")
                    print(f"EasyOCR failed on {image_path.name} {field}: {exc}")
            prediction = normalize_text(" ".join(part for part in parts if part))
        records.append({
            "split": row.split,
            "image_id": row.image_id,
            "target_text": row.target_text,
            "ocr_image": row.ocr_image,
            "prediction": prediction,
        })
    predictions = pd.DataFrame(records)
    predictions.to_csv(PREDICTION_ROOT / "easyocr_predictions.csv", index=False, encoding="utf-8-sig")
    return predictions


easyocr_predictions = run_easyocr(ocr_manifests["test"])
easyocr_eval = evaluate_predictions(easyocr_predictions, "EasyOCR row-wise")
easyocr_eval.to_csv(PREDICTION_ROOT / "easyocr_evaluation.csv", index=False, encoding="utf-8-sig")
display(easyocr_eval.head())


## 5. Baseline OCR: Tesseract Bengali

Tesseract is the baseline comparison. It uses the same DeblurGAN-restored OCR crop images with a simple grayscale, resize, and threshold preprocessing step.


In [ ]:
def preprocess_for_tesseract_array(image, scale=4):
    if image.ndim == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    image = cv2.resize(image, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    image = cv2.GaussianBlur(image, (3, 3), 0)
    image = cv2.equalizeHist(image)
    _, image = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return image


def run_tesseract(frame):
    try:
        import pytesseract
    except Exception as exc:
        print("pytesseract is not available:", exc)
        return empty_predictions(frame, "pytesseract import failed")

    tesseract_config = f"{TESSERACT_CONFIG} -c tessedit_char_whitelist={OCR_ALLOWLIST}"
    records = []
    for row in tqdm(list(frame.itertuples(index=False)), desc="Tesseract row-wise test"):
        image_path = project_path(row.ocr_image)
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            prediction = ""
        else:
            parts = []
            for field in ["top", "bottom"]:
                try:
                    row_image = crop_ocr_row_image(image, field, getattr(row, "row_count", 2))
                    processed = preprocess_for_tesseract_array(row_image)
                    parts.append(pytesseract.image_to_string(processed, lang=TESSERACT_LANG, config=tesseract_config))
                except Exception as exc:
                    parts.append("")
                    print(f"Tesseract failed on {image_path.name} {field}: {exc}")
            prediction = normalize_text(" ".join(parts))
        records.append({
            "split": row.split,
            "image_id": row.image_id,
            "target_text": row.target_text,
            "ocr_image": row.ocr_image,
            "prediction": prediction,
        })
    predictions = pd.DataFrame(records)
    predictions.to_csv(PREDICTION_ROOT / "tesseract_predictions.csv", index=False, encoding="utf-8-sig")
    return predictions


tesseract_predictions = run_tesseract(ocr_manifests["test"])
tesseract_eval = evaluate_predictions(tesseract_predictions, "Tesseract Bengali row-wise")
tesseract_eval.to_csv(PREDICTION_ROOT / "tesseract_evaluation.csv", index=False, encoding="utf-8-sig")
display(tesseract_eval.head())


## 6. YOLOv8 Token OCR on DeblurGAN Plate Crops

This section uses a standard Ultralytics YOLOv8 token detector for OCR. The token OCR dataset is built from DeblurGAN-restored plate crops, so rerunning this section trains and evaluates OCR on the restored input branch.


In [ ]:
YOLO_TOKEN_CLASS_NAMES = [
    "dhaka", "chattro", "sylhet", "metro",
    "ka", "kha", "ga", "gha", "cha", "chha", "ja", "jha", "ta", "tha", "ba", "bha", "m", "na", "sa",
    "0", "1", "2", "3", "4", "5", "6", "7", "8", "9",
]
YOLO_TOKEN_TO_BANGLA = {
    "dhaka": TARGET_TOKEN_ALIASES["dhaka"],
    "chattro": TARGET_TOKEN_ALIASES["chattro"],
    "sylhet": TARGET_TOKEN_ALIASES["sylhet"],
    "metro": TARGET_TOKEN_ALIASES["metro"],
    "ka": TARGET_TOKEN_ALIASES["ka"],
    "kha": TARGET_TOKEN_ALIASES["kha"],
    "ga": TARGET_TOKEN_ALIASES["ga"],
    "gha": TARGET_TOKEN_ALIASES["gha"],
    "cha": TARGET_TOKEN_ALIASES["cha"],
    "chha": TARGET_TOKEN_ALIASES["chha"],
    "ja": TARGET_TOKEN_ALIASES["ja"],
    "jha": TARGET_TOKEN_ALIASES["jha"],
    "ta": TARGET_TOKEN_ALIASES["ta"],
    "tha": TARGET_TOKEN_ALIASES["tha"],
    "ba": TARGET_TOKEN_ALIASES["ba"],
    "bha": TARGET_TOKEN_ALIASES["bha"],
    "m": TARGET_TOKEN_ALIASES["m"],
    "na": TARGET_TOKEN_ALIASES["na"],
    "sa": TARGET_TOKEN_ALIASES["sa"],
    **{str(index): BANGLA_DIGITS[str(index)] for index in range(10)},
}
BANGLA_TO_YOLO_TOKEN = {bangla: token for token, bangla in YOLO_TOKEN_TO_BANGLA.items()}
YOLO_TOKEN_TO_INDEX = {name: index for index, name in enumerate(YOLO_TOKEN_CLASS_NAMES)}
YOLO_INDEX_TO_BANGLA = {index: YOLO_TOKEN_TO_BANGLA[name] for index, name in enumerate(YOLO_TOKEN_CLASS_NAMES)}


def source_image_shape(image_path):
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(f"Could not read source image: {image_path}")
    height, width = image.shape[:2]
    return width, height


def mapped_token_boxes_for_row(row):
    source_image = project_path(row.source_image)
    source_label = project_path(row.source_label)
    image_width, image_height = source_image_shape(source_image)
    crop_x1, crop_y1, crop_x2, crop_y2 = map(float, [row.x1, row.y1, row.x2, row.y2])
    crop_width = max(1.0, crop_x2 - crop_x1)
    crop_height = max(1.0, crop_y2 - crop_y1)
    boxes = []

    for item in read_character_tokens(source_label):
        token_name = BANGLA_TO_YOLO_TOKEN.get(item["token"])
        if token_name is None:
            continue
        abs_x = float(item["x_center"]) * image_width
        abs_y = float(item["y_center"]) * image_height
        abs_w = float(item["width"]) * image_width
        abs_h = float(item["height"]) * image_height
        abs_x1, abs_y1 = abs_x - abs_w / 2, abs_y - abs_h / 2
        abs_x2, abs_y2 = abs_x + abs_w / 2, abs_y + abs_h / 2

        x1 = (abs_x1 - crop_x1) / crop_width
        y1 = (abs_y1 - crop_y1) / crop_height
        x2 = (abs_x2 - crop_x1) / crop_width
        y2 = (abs_y2 - crop_y1) / crop_height
        x1, y1 = max(0.0, x1), max(0.0, y1)
        x2, y2 = min(1.0, x2), min(1.0, y2)
        if x2 <= x1 or y2 <= y1:
            continue

        boxes.append({
            "class_index": YOLO_TOKEN_TO_INDEX[token_name],
            "token": item["token"],
            "x_center": (x1 + x2) / 2,
            "y_center": (y1 + y2) / 2,
            "width": x2 - x1,
            "height": y2 - y1,
        })
    return boxes


def write_yolo_token_dataset(manifests):
    dataset_root = YOLO_TOKEN_DATASET_ROOT
    data_yaml = dataset_root / "data.yaml"
    total_boxes = 0
    split_counts = {}

    for split, frame in manifests.items():
        image_dir = dataset_root / split / "images"
        label_dir = dataset_root / split / "labels"
        image_dir.mkdir(parents=True, exist_ok=True)
        label_dir.mkdir(parents=True, exist_ok=True)
        split_boxes = 0

        for row in frame.itertuples(index=False):
            source_image_path = project_path(row.ocr_image)
            if not source_image_path.exists():
                continue
            image_name = f"{row.image_id}_v{int(row.variant)}.png"
            target_image_path = image_dir / image_name
            if not target_image_path.exists() or OVERWRITE_YOLO_TOKEN_DATASET_IMAGES:
                shutil.copy2(source_image_path, target_image_path)

            boxes = mapped_token_boxes_for_row(row)
            label_lines = [
                f"{box['class_index']} {box['x_center']:.6f} {box['y_center']:.6f} {box['width']:.6f} {box['height']:.6f}"
                for box in boxes
            ]
            (label_dir / f"{Path(image_name).stem}.txt").write_text("\n".join(label_lines), encoding="utf-8")
            split_boxes += len(label_lines)

        split_counts[split] = {"images": len(frame), "boxes": split_boxes}
        total_boxes += split_boxes

    names_block = "\n".join(f"  {index}: {name}" for index, name in enumerate(YOLO_TOKEN_CLASS_NAMES))
    data_yaml.write_text(
        "\n".join([
            f"path: {dataset_root.as_posix()}",
            "train: train/images",
            "val: valid/images",
            "test: test/images",
            f"nc: {len(YOLO_TOKEN_CLASS_NAMES)}",
            "names:",
            names_block,
            "",
        ]),
        encoding="utf-8",
    )
    print("YOLO token dataset:", dataset_root)
    print("YOLO token split counts:", split_counts)
    print("YOLO token boxes:", total_boxes)
    return data_yaml


def load_ultralytics_yolo():
    os.environ.setdefault("YOLO_CONFIG_DIR", str(ULTRALYTICS_CONFIG_DIR))
    from ultralytics import YOLO
    return YOLO


def train_or_load_yolo_token_model(data_yaml):
    YOLO = load_ultralytics_yolo()
    if YOLO_TOKEN_WEIGHTS.exists() and not RETRAIN_YOLO_TOKEN_OCR:
        print("Loading existing YOLO token OCR weights:", YOLO_TOKEN_WEIGHTS)
        return YOLO(str(YOLO_TOKEN_WEIGHTS))
    if not RUN_YOLO_TOKEN_OCR_TRAINING:
        print("YOLO token OCR training is disabled and no weights were loaded.")
        return None

    base_model = YOLO_TOKEN_BASE_MODEL if YOLO_TOKEN_BASE_MODEL.exists() else "yolov8s.pt"
    model = YOLO(str(base_model))
    model.train(
        data=str(data_yaml),
        imgsz=YOLO_TOKEN_IMAGE_SIZE,
        epochs=YOLO_TOKEN_EPOCHS,
        batch=YOLO_TOKEN_BATCH_SIZE,
        patience=20,
        project=str(YOLO_TOKEN_RUN_ROOT),
        name=YOLO_TOKEN_RUN_NAME,
        exist_ok=True,
        device=0 if torch.cuda.is_available() else "cpu",
        workers=0,
        seed=SEED,
    )
    return YOLO(str(YOLO_TOKEN_WEIGHTS))


def detections_from_yolo_result(result):
    detections = []
    if result is None or result.boxes is None:
        return detections
    xyxy = result.boxes.xyxy.detach().cpu().numpy()
    classes = result.boxes.cls.detach().cpu().numpy().astype(int)
    confidences = result.boxes.conf.detach().cpu().numpy()
    for box, class_index, confidence in zip(xyxy, classes, confidences):
        if class_index not in YOLO_INDEX_TO_BANGLA:
            continue
        x1, y1, x2, y2 = map(float, box)
        token = YOLO_INDEX_TO_BANGLA[int(class_index)]
        detections.append({
            "token": token,
            "confidence": float(confidence),
            "x_center": (x1 + x2) / 2,
            "y_center": (y1 + y2) / 2,
            "width": x2 - x1,
            "height": y2 - y1,
            "is_digit": token in BANGLA_DIGIT_VALUES,
            "is_city": token in CITY_TOKENS,
            "is_series": token in SERIES_TOKENS,
            "is_metro": token == METRO_TOKEN,
        })
    return detections


def best_detection(detections, predicate):
    candidates = [item for item in detections if predicate(item)]
    if not candidates:
        return None
    return max(candidates, key=lambda item: item["confidence"])


def dedupe_digit_detections(digits, min_distance=8.0):
    digits = sorted(digits, key=lambda item: item["confidence"], reverse=True)
    kept = []
    for digit in digits:
        if all(abs(digit["x_center"] - other["x_center"]) >= min_distance for other in kept):
            kept.append(digit)
    return sorted(kept, key=lambda item: item["x_center"])


def decode_yolo_token_detections(detections):
    city = best_detection(detections, lambda item: item["is_city"])
    metro = best_detection(detections, lambda item: item["is_metro"])
    series = best_detection(detections, lambda item: item["is_series"])
    top_parts = []
    if city is not None:
        top_parts.append(city["token"])
    if metro is not None:
        top_parts.append(METRO_TOKEN)
    if series is not None:
        top_parts.append(series["token"])

    digit_detections = dedupe_digit_detections([item for item in detections if item["is_digit"]])
    if len(digit_detections) > BOTTOM_DIGIT_COUNT:
        digit_detections = sorted(digit_detections, key=lambda item: item["confidence"], reverse=True)[:BOTTOM_DIGIT_COUNT]
        digit_detections = sorted(digit_detections, key=lambda item: item["x_center"])
    bottom_text = "".join(item["token"] for item in digit_detections)
    if len(bottom_text) != BOTTOM_DIGIT_COUNT:
        bottom_text = enforce_yolo_digit_count(bottom_text)

    return " ".join(part for part in [" ".join(top_parts).strip(), bottom_text] if part).strip()


def enforce_yolo_digit_count(text):
    digits = "".join(char for char in normalize_text(text) if char in BANGLA_DIGIT_VALUES)
    if len(digits) == BOTTOM_DIGIT_COUNT:
        return digits
    if len(digits) > BOTTOM_DIGIT_COUNT:
        return digits[:BOTTOM_DIGIT_COUNT]
    return digits


def run_yolo_token_ocr(model, frame):
    if model is None:
        return empty_predictions(frame, "YOLO token OCR model unavailable")

    records = []
    for row in tqdm(list(frame.itertuples(index=False)), desc="YOLO token OCR test"):
        image_path = project_path(row.ocr_image)
        results = model.predict(
            source=str(image_path),
            imgsz=YOLO_TOKEN_IMAGE_SIZE,
            conf=YOLO_TOKEN_CONFIDENCE,
            iou=YOLO_TOKEN_IOU,
            verbose=False,
        )
        detections = detections_from_yolo_result(results[0] if results else None)
        prediction = decode_yolo_token_detections(detections)
        records.append({
            "split": row.split,
            "image_id": row.image_id,
            "target_text": row.target_text,
            "ocr_image": row.ocr_image,
            "prediction": normalize_text(prediction),
            "detected_tokens": " ".join(item["token"] for item in sorted(detections, key=lambda value: (value["y_center"], value["x_center"]))),
            "detection_count": len(detections),
        })
    predictions = pd.DataFrame(records)
    predictions.to_csv(PREDICTION_ROOT / "yolo_token_ocr_predictions.csv", index=False, encoding="utf-8-sig")
    return predictions


yolo_token_data_yaml = write_yolo_token_dataset(ocr_manifests)
yolo_token_model = train_or_load_yolo_token_model(yolo_token_data_yaml)
yolo_token_predictions = run_yolo_token_ocr(yolo_token_model, ocr_manifests["test"])
yolo_token_eval = evaluate_predictions(yolo_token_predictions, "YOLOv8 token OCR")
yolo_token_eval.to_csv(PREDICTION_ROOT / "yolo_token_ocr_evaluation.csv", index=False, encoding="utf-8-sig")
display(yolo_token_eval.head())


In [ ]:
print("Custom PyTorch OCR model is disabled. The notebook now uses YOLOv8 token OCR on DeblurGAN-restored plate crops as the main OCR method.")
print("DeblurGAN OCR input root:", DEBLURRED_OCR_INPUT_ROOT)
print("YOLO token OCR weights:", YOLO_TOKEN_WEIGHTS)
print("Retrain YOLO token OCR:", RETRAIN_YOLO_TOKEN_OCR)


## 7. Final Metrics and Figures

This section summarizes the OCR engines that ran above. The main result is `YOLOv8 token OCR`, evaluated on DeblurGAN-restored plate crops.


In [ ]:
evaluation_frames = []
for name in ["yolo_token_eval", "easyocr_eval", "tesseract_eval"]:
    frame = globals().get(name)
    if isinstance(frame, pd.DataFrame) and not frame.empty:
        evaluation_frames.append(frame)

if evaluation_frames:
    all_evaluations = pd.concat(evaluation_frames, ignore_index=True)
else:
    all_evaluations = pd.DataFrame(columns=["method", "split", "image_id", "target_text", "prediction", "cer", "wer", "exact_match", "ocr_image"])

all_evaluations.to_csv(PREDICTION_ROOT / "all_ocr_predictions_or_evaluations.csv", index=False, encoding="utf-8-sig")

if HAS_OCR_GROUND_TRUTH and not all_evaluations.empty:
    summary = (
        all_evaluations
        .groupby("method", as_index=False)
        .agg(samples=("image_id", "count"), cer=("cer", "mean"), wer=("wer", "mean"), exact_plate_match_accuracy=("exact_match", "mean"))
        .sort_values(["cer", "wer", "method"])
    )
    summary.to_csv(PREDICTION_ROOT / "ocr_metric_summary.csv", index=False, encoding="utf-8-sig")
    display(summary)
else:
    summary = pd.DataFrame()
    print("OCR ground truth is not available. Saved OCR predictions, but CER/WER/accuracy are skipped.")


In [ ]:
def plot_metric_summary(summary_frame):
    figure, axes = plt.subplots(1, 3, figsize=(15, 4))
    metrics = [
        ("cer", "CER", "lower is better"),
        ("wer", "WER", "lower is better"),
        ("exact_plate_match_accuracy", "Exact Plate Match Accuracy", "higher is better"),
    ]
    colors = ["#3B82F6", "#10B981", "#F59E0B"]
    for axis, (column, title, subtitle), color in zip(axes, metrics, colors):
        axis.bar(summary_frame["method"], summary_frame[column], color=color)
        axis.set_title(f"{title}\n{subtitle}")
        axis.set_ylim(0, 1 if column != "exact_plate_match_accuracy" else max(1, summary_frame[column].max() * 1.15))
        axis.tick_params(axis="x", rotation=25)
    figure.tight_layout()
    output_path = FIGURE_ROOT / "ocr_metric_summary.png"
    figure.savefig(output_path, dpi=200, bbox_inches="tight")
    print("Saved:", output_path)
    plt.show()


plot_metric_summary(summary)



In [ ]:
def show_prediction_examples(evaluations, count=8):
    if evaluations.empty:
        print("No evaluated predictions to show.")
        return
    sample = evaluations.sample(min(count, len(evaluations)), random_state=19)
    figure, axes = plt.subplots(len(sample), 2, figsize=(12, 2.8 * len(sample)))
    if len(sample) == 1:
        axes = np.array([axes])
    for axis_row, (_, row) in zip(axes, sample.iterrows()):
        first = all_evaluations[
            (all_evaluations["method"] == row["method"]) &
            (all_evaluations["image_id"] == row["image_id"])
        ].iloc[0]
        image = cv2.imread(str(project_path(first["ocr_image"])))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        axis_row[0].imshow(image)
        axis_row[0].set_title("DeblurGAN OCR input crop")
        axis_row[0].axis("off")
        text = f"Method: {row['method']}\nTarget: {row['target_text']}\nPrediction: {row['prediction']}\nExact: {row['exact_match']}"
        axis_row[1].text(0.02, 0.5, text, fontsize=11, va="center", wrap=True)
        axis_row[1].axis("off")
    plt.tight_layout()
    output_path = FIGURE_ROOT / "ocr_prediction_examples.png"
    plt.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved prediction examples:", output_path)


show_prediction_examples(all_evaluations)


## Saved Outputs

- DeblurGAN OCR input crops: `runs/ocr_text_recognition_psnr_v2/deblurred_ocr_inputs/`
- OCR manifests for Bengali-labeled DeblurGAN crops: `runs/ocr_text_recognition_psnr_v2/manifests/`
- YOLO token OCR dataset: `runs/ocr_text_recognition_psnr_v2/yolo_token_dataset/`
- YOLO token OCR weights: `runs/ocr_text_recognition_psnr_v2/yolo_token_ocr/yolov8_token_detector/weights/best.pt`
- YOLO token OCR predictions/evaluation: `runs/ocr_text_recognition_psnr_v2/predictions/yolo_token_ocr_predictions.csv` and `yolo_token_ocr_evaluation.csv`
- Per-model predictions and evaluations: `runs/ocr_text_recognition_psnr_v2/predictions/`
- OCR metric and qualitative figures: `runs/ocr_text_recognition_psnr_v2/figures/`
